# 🌀 Notebook 1: The Saga Pattern

A single business transaction spans **multiple services** — e.g. `order-service` reserves stock, `payment-service` charges, `shipping-service` books a courier.
Classic ACID with 2-phase commit doesn't scale across services.

A **saga** is a sequence of local transactions. If a later step fails, earlier steps are undone by **compensating transactions**.

> *Put another way*: you can't use 'undo' on separate databases, so you bake *reverse actions* into the workflow.

## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## A tiny linear saga

In [ ]:
class Step:
    def __init__(self, name, do, undo):
        self.name, self.do, self.undo = name, do, undo

def run_saga(steps):
    done = []
    try:
        for s in steps:
            print(f'→ {s.name}')
            s.do()
            done.append(s)
    except Exception as e:
        print(f'✗ failure in {s.name}: {e}. Compensating...')
        for d in reversed(done):
            print(f'  ↶ undo {d.name}')
            d.undo()
        raise

# Business actions (in-memory stand-ins)
state = {'stock': 10, 'charged': 0, 'shipment': None}

def reserve_stock(): state['stock'] -= 1
def release_stock(): state['stock'] += 1

def charge(): state['charged'] = 20
def refund(): state['charged'] = 0

def ship(): state['shipment'] = 'booked'
def cancel_ship(): state['shipment'] = None

steps = [
    Step('reserve_stock', reserve_stock, release_stock),
    Step('charge',        charge,         refund),
    Step('ship',          ship,           cancel_ship),
]
run_saga(steps)
print('state:', state)
